# Modeling — Texas / Houston price prediction

Uses `src/train.py`:

- **Target:** `log1p(price)` for stability  
- **Models:** Linear Regression, Random Forest (GridSearchCV), XGBoost or GradientBoosting fallback  
- **Validation:** 5-fold CV inside grid search; hold-out test split  
- **Artifact:** `models/pipeline.pkl` (best estimator + feature column list)  
- **Metrics:** `models/metrics.json`

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FEAT = PROJECT_ROOT / "data/processed/texas_houston_features.csv"
df = pd.read_csv(FEAT)
print("Features shape:", df.shape)
df[["price", "sqft", "bedrooms", "bathrooms"]].describe()

## Train & compare models (writes `models/pipeline.pkl`)

In [ ]:
from src.train import train_models

model_path, metrics_path = train_models()
print("Model:", model_path)
print("Metrics:", metrics_path)

## Load metrics summary

In [ ]:
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
rows = [{"model": k, **{kk: vv for kk, vv in v.items() if kk in ("r2", "rmse", "mae")}} for k, v in metrics.items() if k != "best_model" and isinstance(v, dict)]
pd.DataFrame(rows).sort_values("r2", ascending=False)

## Optional: quick sanity prediction (single row)
Requires `joblib` and trained `models/pipeline.pkl`.

In [ ]:
import joblib
import numpy as np

pkg = joblib.load(PROJECT_ROOT / "models/pipeline.pkl")
pipe, cols = pkg["pipeline"], pkg["feature_columns"]
row = df.iloc[[0]][cols]
pred = float(np.expm1(pipe.predict(row)[0]))
print("Example pred $", round(pred, 2), "| actual $", float(df.iloc[0]["price"]))